# 📊 Notebook 3 — Clasificadores RNN (LSTM, BiLSTM, GRU, SimpleRNN)
### Sistema Ernesto Investing AI — Apoyo en Decisiones de Inversión usando IA

---

| Campo | Detalle |
|---|---|
| **Universidad** | Universidad Nacional Mayor de San Marcos (UNMSM) |
| **Facultad** | Facultad de Ingeniería de Sistemas e Informática (FISI) |
| **Curso** | Introducción al Desarrollo de Software (iDeSo) |
| **Docente** | Prof. Mg. Ing. Ernesto D. Cancho-Rodríguez |
| **Entregable** | Laboratorio — Semana 11 — Notebook 3 |
| **Tema** | Desarrollo del Backend con IA |
| **Grupo 2** | Abigail Aroste, Adrian Garcia, Alonso Huayta, Diego Jimenez, Gianpiere Alvarado |

---

## 🎯 Objetivo

Implementar y comparar **4 arquitecturas de redes neuronales recurrentes** (LSTM, BiLSTM, GRU,
SimpleRNN) para la **clasificación binaria de tendencias bursátiles** (BUY / SELL) de 5 activos
financieros mineros con operaciones en Perú, usando **datos reales** descargados de Yahoo Finance
mediante `yfinance`.

El notebook produce, en su celda final, el archivo **`datos_rnns_clf.json`**, que constituye el
**contrato de datos (data contract)** entre este backend en Python y el frontend
`ernesto_investing_rnns_clasificador.html` (Anexo C.3 de las especificaciones).

## 🪙 Activos financieros del estudio

| Ticker | Empresa | Bolsa |
|---|---|---|
| `FSM` | Fortuna Silver Mines Inc. | NYSE |
| `VOLCABC1.LM` | Volcan Compañía Minera S.A.A. | BVL |
| `ABX.TO` | Barrick Gold Corporation | TSX |
| `BVN` | Compañía de Minas Buenaventura S.A.A. | NYSE |
| `BHP` | BHP Billiton Limited | NYSE |

## 🧠 Arquitecturas de referencia (investigación doctoral del profesor)

| Modelo | Arquitectura | Épocas | Batch |
|---|---|---|---|
| LSTM | 2 capas LSTM (260 → 130) + Dropout(0.2) + Dense(1, sigmoid) | 80 | 64 |
| BiLSTM | 2 capas Bidirectional LSTM (200 → 100) + Dropout(0.3) + Dense(1, sigmoid) | 80 | 64 |
| GRU | 2 capas GRU (280 → 140) + Dense(1, sigmoid) — *sin Dropout* | 80 | 64 |
| SimpleRNN | 2 capas SimpleRNN (180 → 90) + Dense(1, sigmoid) | 80 | 64 |

> ⚠️ **Importante:** Este notebook usa **únicamente datos reales** de Yahoo Finance. Nunca se
> inventan datos. Si Colab asigna GPU (Runtime → Change runtime type → GPU), el entrenamiento de
> los 20 modelos (5 tickers × 4 arquitecturas) será considerablemente más rápido.


## Módulo 0 · Instalación de librerías

In [ ]:
# Instalación de librerías necesarias (Google Colab gratuito)
!pip install yfinance ta scikit-learn --quiet
print("✅ Librerías instaladas correctamente.")


  Preparing metadata (setup.py) ... done
✅ Librerías instaladas correctamente.


## Módulo 1 · Importaciones y configuración de reproducibilidad

In [ ]:
# Importaciones generales
import json
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import yfinance as yf
import ta

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Bidirectional, GRU, SimpleRNN, Dense, Dropout
from tensorflow.keras.optimizers import Adam

warnings.filterwarnings("ignore")

# Semilla fija para reproducibilidad (requisito de las especificaciones)
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("✅ Librerías importadas. TensorFlow versión:", tf.__version__)


✅ Librerías importadas. TensorFlow versión: 2.20.0


## Módulo 2 · Configuración de tickers, ventana temporal y arquitecturas

Las claves de `TICKERS` son exactamente los símbolos de Yahoo Finance usados por el selector del
frontend `ernesto_investing_rnns_clasificador.html`, de modo que el JSON generado calce sin
transformaciones adicionales.

In [ ]:
# Configuración de los 5 activos del estudio (mismos símbolos que usa el frontend HTML)
TICKERS = ['FSM', 'VOLCABC1.LM', 'ABX.TO', 'BVN', 'BHP']

# Periodo de descarga de datos reales
PERIODO = "2y"     # últimos 2 años de datos OHLCV reales
INTERVALO = "1d"   # velas diarias

# Ventana temporal (días) usada para construir las secuencias de entrada de las RNN
WINDOW_SIZE = 20

# Hiperparámetros de entrenamiento (idénticos para las 4 arquitecturas, según especificaciones)
EPOCAS = 80
BATCH_SIZE = 64
LEARNING_RATE = 0.001
TEST_RATIO = 0.20  # 80/20, respetando el orden temporal (sin mezclar)

# Descripción y color de cada arquitectura — el color coincide con la paleta del frontend
# (variables CSS --lstm, --bilstm, --gru, --srnn) para que la UI no requiera cambios.
MODEL_SPECS = {
    "LSTM": {
        "arq": "LSTM(260)→Drop(0.2)→LSTM(130)→Dense(1)",
        "color": "#1565c0",
        "dropout": True,
    },
    "BiLSTM": {
        "arq": "BiLSTM(200)→Drop(0.3)→BiLSTM(100)→Dense(1)",
        "color": "#6a1b9a",
        "dropout": True,
    },
    "GRU": {
        "arq": "GRU(280)→GRU(140)→Dense(1)",
        "color": "#e65100",
        "dropout": False,
    },
    "SimpleRNN": {
        "arq": "RNN(180)→RNN(90)→Dense(1)",
        "color": "#00838f",
        "dropout": False,
    },
}

print("Tickers configurados:", TICKERS)
print("Modelos a entrenar por ticker:", list(MODEL_SPECS.keys()))


Tickers configurados: ['FSM', 'VOLCABC1.LM', 'ABX.TO', 'BVN', 'BHP']
Modelos a entrenar por ticker: ['LSTM', 'BiLSTM', 'GRU', 'SimpleRNN']


## Módulo 3 · Ingesta de datos reales con `yfinance`

In [ ]:
def descargar_datos(ticker: str, periodo: str = PERIODO, intervalo: str = INTERVALO) -> pd.DataFrame:
    '''Descarga datos OHLCV reales de Yahoo Finance para un ticker dado.

    Incluye manejo de errores: si la descarga falla o devuelve un DataFrame vacío,
    se levanta una excepción controlada para que el ticker pueda omitirse en el
    bucle principal sin detener la ejecución de todo el notebook.
    '''
    try:
        df = yf.download(ticker, period=periodo, interval=intervalo, progress=False, auto_adjust=True)
        if df is None or df.empty:
            raise ValueError(f"Yahoo Finance no devolvió datos para {ticker}.")

        # yfinance puede devolver columnas como MultiIndex si se piden varios tickers;
        # aquí forzamos a un único nivel por seguridad.
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)

        df = df[["Open", "High", "Low", "Close", "Volume"]].dropna()
        df.index = pd.to_datetime(df.index)
        return df
    except Exception as e:
        print(f"⚠️  Error al descargar {ticker}: {e}")
        return pd.DataFrame()


# Descarga real de los 5 tickers del estudio
datos_crudos = {}
for tk in TICKERS:
    df_tk = descargar_datos(tk)
    if not df_tk.empty:
        datos_crudos[tk] = df_tk
        print(f"✅ {tk}: {len(df_tk)} velas diarias descargadas "
              f"({df_tk.index.min().date()} → {df_tk.index.max().date()})")
    else:
        print(f"❌ {tk}: sin datos disponibles, se omitirá del análisis.")


✅ FSM: 501 velas diarias descargadas (2024-06-20 → 2026-06-18)
✅ VOLCABC1.LM: 494 velas diarias descargadas (2024-06-20 → 2026-06-18)
✅ ABX.TO: 503 velas diarias descargadas (2024-06-19 → 2026-06-19)
✅ BVN: 501 velas diarias descargadas (2024-06-20 → 2026-06-18)
✅ BHP: 501 velas diarias descargadas (2024-06-20 → 2026-06-18)


## Módulo 4 · Ingeniería de características y variable objetivo (Trend)

Se calculan los indicadores técnicos estándar (SMA, EMA, RSI, MACD, Bollinger Bands) y se construye
la variable objetivo binaria **`Trend`**: `1` (BUY) si el retorno del día siguiente es positivo,
`0` (SELL) si es negativo — exactamente como especifica la sección 3.2 del documento de trabajo.

In [ ]:
def calcular_indicadores(df: pd.DataFrame) -> pd.DataFrame:
    '''Calcula indicadores técnicos y la variable objetivo binaria Trend (BUY=1 / SELL=0).'''
    data = df.copy()

    # Retornos diarios
    data["Return"] = data["Close"].pct_change()

    # Medias móviles
    data["SMA_20"] = ta.trend.sma_indicator(data["Close"], window=20)
    data["EMA_12"] = ta.trend.ema_indicator(data["Close"], window=12)

    # RSI (14 periodos)
    data["RSI_14"] = ta.momentum.rsi(data["Close"], window=14)

    # MACD (línea, señal, histograma)
    macd = ta.trend.MACD(data["Close"])
    data["MACD"] = macd.macd()
    data["MACD_signal"] = macd.macd_signal()
    data["MACD_hist"] = macd.macd_diff()

    # Bandas de Bollinger
    bb = ta.volatility.BollingerBands(data["Close"], window=20, window_dev=2)
    data["BB_high"] = bb.bollinger_hband()
    data["BB_low"] = bb.bollinger_lband()
    data["BB_width"] = bb.bollinger_wband()

    # Volatilidad (desviación estándar móvil de los retornos)
    data["Volatility_10"] = data["Return"].rolling(window=10).std()

    # Variable objetivo binaria: 1=BUY (sube al día siguiente), 0=SELL (baja al día siguiente)
    data["Trend"] = (data["Close"].shift(-1) > data["Close"]).astype(int)

    data = data.dropna().reset_index().rename(columns={"index": "Date", "Date": "Date"})
    return data


FEATURE_COLS = [
    "Open", "High", "Low", "Close", "Volume", "Return",
    "SMA_20", "EMA_12", "RSI_14", "MACD", "MACD_signal", "MACD_hist",
    "BB_high", "BB_low", "BB_width", "Volatility_10",
]

datos_features = {}
for tk, df_tk in datos_crudos.items():
    df_ind = calcular_indicadores(df_tk)
    datos_features[tk] = df_ind
    print(f"{tk}: {len(df_ind)} filas válidas tras calcular indicadores "
          f"(BUY={int(df_ind['Trend'].sum())} / SELL={int((1-df_ind['Trend']).sum())})")


FSM: 468 filas válidas tras calcular indicadores (BUY=244 / SELL=224)
VOLCABC1.LM: 461 filas válidas tras calcular indicadores (BUY=211 / SELL=250)
ABX.TO: 470 filas válidas tras calcular indicadores (BUY=259 / SELL=211)
BVN: 468 filas válidas tras calcular indicadores (BUY=247 / SELL=221)
BHP: 468 filas válidas tras calcular indicadores (BUY=252 / SELL=216)


## Módulo 5 · Normalización y construcción de secuencias temporales

In [ ]:
def crear_secuencias(df: pd.DataFrame, feature_cols: list, window: int = WINDOW_SIZE):
    '''Normaliza las características con MinMaxScaler y construye secuencias deslizantes
    de tamaño `window` para alimentar a las redes recurrentes.

    Retorna X (n_secuencias, window, n_features), y (n_secuencias,), el scaler ajustado
    y las fechas/precios de cierre alineados a cada secuencia (para reconstruir señales).
    '''
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(df[feature_cols].values)

    X, y, fechas_seq, precios_seq = [], [], [], []
    for i in range(window, len(df)):
        X.append(X_scaled[i - window:i])
        y.append(df["Trend"].iloc[i])
        fechas_seq.append(df["Date"].iloc[i])
        precios_seq.append(df["Close"].iloc[i])

    return (np.array(X), np.array(y), scaler,
            pd.to_datetime(pd.Series(fechas_seq)), np.array(precios_seq))


def split_temporal(X, y, fechas, precios, test_ratio: float = TEST_RATIO):
    '''Particiona train/test respetando el orden temporal (sin mezclar, sin fuga de datos).'''
    n_test = int(len(X) * test_ratio)
    n_train = len(X) - n_test
    return (X[:n_train], X[n_train:], y[:n_train], y[n_train:],
            fechas[n_train:], precios[n_train:])


# Construcción de secuencias y partición train/test por ticker
datos_secuencias = {}
for tk, df_ind in datos_features.items():
    X, y, scaler, fechas_seq, precios_seq = crear_secuencias(df_ind, FEATURE_COLS)
    X_train, X_test, y_train, y_test, fechas_test, precios_test = split_temporal(X, y, fechas_seq, precios_seq)
    datos_secuencias[tk] = {
        "X_train": X_train, "X_test": X_test,
        "y_train": y_train, "y_test": y_test,
        "fechas_full": fechas_seq, "precios_full": precios_seq,
        "fechas_test": fechas_test, "precios_test": precios_test,
        "X_full": X,
    }
    print(f"{tk}: {len(X_train)} secuencias train / {len(X_test)} secuencias test "
          f"(ventana={WINDOW_SIZE} días, {X.shape[2]} características)")


FSM: 359 secuencias train / 89 secuencias test (ventana=20 días, 16 características)
VOLCABC1.LM: 353 secuencias train / 88 secuencias test (ventana=20 días, 16 características)
ABX.TO: 360 secuencias train / 90 secuencias test (ventana=20 días, 16 características)
BVN: 359 secuencias train / 89 secuencias test (ventana=20 días, 16 características)
BHP: 359 secuencias train / 89 secuencias test (ventana=20 días, 16 características)


## Módulo 6 · Definición de las 4 arquitecturas RNN

Cada función construye exactamente la arquitectura especificada en el documento de trabajo
(sección 4.1.3 y Anexos A.3–A.6).

In [ ]:
def build_lstm(input_shape):
    model = Sequential([
        LSTM(260, return_sequences=True, input_shape=input_shape),
        Dropout(0.2),
        LSTM(130),
        Dense(1, activation="sigmoid"),
    ])
    model.compile(optimizer=Adam(learning_rate=LEARNING_RATE),
                   loss="binary_crossentropy", metrics=["accuracy"])
    return model


def build_bilstm(input_shape):
    model = Sequential([
        Bidirectional(LSTM(200, return_sequences=True), input_shape=input_shape),
        Dropout(0.3),
        Bidirectional(LSTM(100)),
        Dense(1, activation="sigmoid"),
    ])
    model.compile(optimizer=Adam(learning_rate=LEARNING_RATE),
                   loss="binary_crossentropy", metrics=["accuracy"])
    return model


def build_gru(input_shape):
    # Diseño intencional: sin capas de Dropout, según especificaciones (Anexo A.5)
    model = Sequential([
        GRU(280, return_sequences=True, input_shape=input_shape),
        GRU(140),
        Dense(1, activation="sigmoid"),
    ])
    model.compile(optimizer=Adam(learning_rate=LEARNING_RATE),
                   loss="binary_crossentropy", metrics=["accuracy"])
    return model


def build_simplernn(input_shape):
    # Arquitectura baseline para comparar contra modelos con compuertas (LSTM/GRU)
    model = Sequential([
        SimpleRNN(180, return_sequences=True, input_shape=input_shape),
        SimpleRNN(90),
        Dense(1, activation="sigmoid"),
    ])
    model.compile(optimizer=Adam(learning_rate=LEARNING_RATE),
                   loss="binary_crossentropy", metrics=["accuracy"])
    return model


MODEL_BUILDERS = {
    "LSTM": build_lstm,
    "BiLSTM": build_bilstm,
    "GRU": build_gru,
    "SimpleRNN": build_simplernn,
}

print("✅ 4 arquitecturas RNN definidas:", list(MODEL_BUILDERS.keys()))


✅ 4 arquitecturas RNN definidas: ['LSTM', 'BiLSTM', 'GRU', 'SimpleRNN']


## Módulo 7 · Función de entrenamiento y evaluación

In [ ]:
def entrenar_y_evaluar(nombre_modelo: str, seq: dict):
    '''Entrena una arquitectura RNN sobre las secuencias de un ticker y calcula sus métricas.

    Retorna un diccionario con: accuracy, precision, recall, f1, matriz de confusión,
    historial de accuracy por época (accHist) y el modelo entrenado (para generar señales
    sobre la serie completa).
    '''
    input_shape = (seq["X_train"].shape[1], seq["X_train"].shape[2])
    model = MODEL_BUILDERS[nombre_modelo](input_shape)

    history = model.fit(
        seq["X_train"], seq["y_train"],
        epochs=EPOCAS, batch_size=BATCH_SIZE,
        validation_data=(seq["X_test"], seq["y_test"]),
        verbose=0,
    )

    y_pred_proba = model.predict(seq["X_test"], verbose=0).ravel()
    y_pred = (y_pred_proba >= 0.5).astype(int)
    y_test = seq["y_test"]

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred).tolist()

    acc_hist = [round(float(v), 4) for v in history.history.get("accuracy", [])]
    # Asegurar longitud == EPOCAS (por si el entrenamiento se detiene antes)
    if len(acc_hist) < EPOCAS:
        acc_hist += [acc_hist[-1] if acc_hist else 0.5] * (EPOCAS - len(acc_hist))

    return {
        "model": model,
        "acc": round(float(acc), 4),
        "prec": round(float(prec), 4),
        "rec": round(float(rec), 4),
        "f1": round(float(f1), 4),
        "confusion_matrix": cm,
        "accHist": acc_hist,
        "y_pred_proba_test": y_pred_proba,
    }


print("✅ Función de entrenamiento/evaluación lista.")


✅ Función de entrenamiento/evaluación lista.


## Módulo 8 · Entrenamiento de los 4 modelos para los 5 tickers

Por cada ticker se entrenan las 4 arquitecturas (20 entrenamientos en total). Para cada modelo se
genera además el arreglo de señales `BUY`/`SELL` sobre **toda la serie de precios** (no solo el
conjunto de prueba), reutilizando el modelo entrenado para predecir sobre `X_full`. Los primeros
`WINDOW_SIZE` días (sin secuencia previa suficiente) se rellenan repitiendo la primera señal
disponible, de modo que el arreglo de señales tenga la misma longitud que `fechas`/`precios`.

In [ ]:
resultados = {}  # resultados[ticker][modelo] = dict con métricas, señales, etc.

for tk in datos_secuencias.keys():
    print(f"\n{'='*60}\n📈 Entrenando modelos para {tk}\n{'='*60}")
    seq = datos_secuencias[tk]
    resultados[tk] = {}

    for nombre_modelo in MODEL_SPECS.keys():
        print(f"  → Entrenando {nombre_modelo} ({EPOCAS} épocas, batch={BATCH_SIZE})...")
        res = entrenar_y_evaluar(nombre_modelo, seq)

        # Predicción sobre la serie completa para generar señales alineadas a fechas/precios
        proba_full = res["model"].predict(seq["X_full"], verbose=0).ravel()
        senales_full = np.where(proba_full >= 0.5, "BUY", "SELL").tolist()

        # Relleno de los primeros WINDOW_SIZE días sin secuencia disponible
        relleno = [senales_full[0]] * WINDOW_SIZE if senales_full else ["SELL"] * WINDOW_SIZE
        senales_full = relleno + senales_full

        senal_actual = senales_full[-1]
        confianza = float(proba_full[-1]) if senal_actual == "BUY" else float(1 - proba_full[-1])

        resultados[tk][nombre_modelo] = {
            "acc": res["acc"], "prec": res["prec"], "rec": res["rec"], "f1": res["f1"],
            "confusion_matrix": res["confusion_matrix"],
            "accHist": res["accHist"],
            "senales": senales_full,
            "senal": senal_actual,
            "conf": round(confianza, 4),
        }

        print(f"    ✅ {nombre_modelo}: Accuracy={res['acc']:.3f}  F1={res['f1']:.3f}  "
              f"Señal actual={senal_actual} ({confianza*100:.1f}%)")

print("\n🏁 Entrenamiento completo de los 20 modelos (5 tickers × 4 arquitecturas).")



📈 Entrenando modelos para FSM
  → Entrenando LSTM (80 épocas, batch=64)...
    ✅ LSTM: Accuracy=0.517  F1=0.538  Señal actual=SELL (70.8%)
  → Entrenando BiLSTM (80 épocas, batch=64)...
    ✅ BiLSTM: Accuracy=0.449  F1=0.614  Señal actual=BUY (60.8%)
  → Entrenando GRU (80 épocas, batch=64)...
    ✅ GRU: Accuracy=0.472  F1=0.515  Señal actual=SELL (54.8%)
  → Entrenando SimpleRNN (80 épocas, batch=64)...
    ✅ SimpleRNN: Accuracy=0.539  F1=0.568  Señal actual=BUY (99.3%)

📈 Entrenando modelos para VOLCABC1.LM
  → Entrenando LSTM (80 épocas, batch=64)...
    ✅ LSTM: Accuracy=0.455  F1=0.625  Señal actual=BUY (65.2%)
  → Entrenando BiLSTM (80 épocas, batch=64)...
    ✅ BiLSTM: Accuracy=0.455  F1=0.571  Señal actual=BUY (56.3%)
  → Entrenando GRU (80 épocas, batch=64)...
    ✅ GRU: Accuracy=0.523  F1=0.562  Señal actual=BUY (99.8%)
  → Entrenando SimpleRNN (80 épocas, batch=64)...
    ✅ SimpleRNN: Accuracy=0.489  F1=0.430  Señal actual=BUY (64.4%)

📈 Entrenando modelos para ABX.TO
  → En

## Módulo 9 · Resumen comparativo de métricas por ticker

In [ ]:
resumen_filas = []
for tk, modelos in resultados.items():
    for nombre_modelo, m in modelos.items():
        resumen_filas.append({
            "Ticker": tk, "Modelo": nombre_modelo,
            "Accuracy": m["acc"], "Precision": m["prec"],
            "Recall": m["rec"], "F1": m["f1"], "Señal": m["senal"],
        })

df_resumen = pd.DataFrame(resumen_filas)
display(df_resumen.sort_values(["Ticker", "Accuracy"], ascending=[True, False]))


,Ticker,Modelo,Accuracy,Precision,Recall,F1,Señal
11,ABX.TO,SimpleRNN,0.5444,0.5200,0.6047,0.5591,SELL
10,ABX.TO,GRU,0.5222,0.5000,0.9070,0.6446,BUY
9,ABX.TO,BiLSTM,0.5000,0.4875,0.9070,0.6341,BUY
8,ABX.TO,LSTM,0.4778,0.4778,1.0000,0.6466,BUY
16,BHP,LSTM,0.5506,0.5506,1.0000,0.7101,BUY
18,BHP,GRU,0.5281,0.5479,0.8163,0.6557,BUY
17,BHP,BiLSTM,0.4944,0.5256,0.8367,0.6457,BUY
19,BHP,SimpleRNN,0.4494,0.5000,0.6735,0.5739,SELL
15,BVN,SimpleRNN,0.6067,0.6042,0.6444,0.6237,SELL
12,BVN,LSTM,0.5056,0.5060,0.9333,0.6562,BUY


## Módulo 10 · Exportación al contrato de datos JSON

Se construye el JSON con la **estructura exacta** que consume el frontend
`ernesto_investing_rnns_clasificador.html`. Para cada ticker se incluyen las claves `fechas`,
`precios` y `modelos` — esta última, una lista de 4 objetos (uno por arquitectura) con los campos
`id`, `arq`, `color`, `acc`, `prec`, `rec`, `f1`, `accHist`, `senales`, `senal` y `conf`,
replicando uno a uno los campos que el JavaScript del HTML espera leer (ver función
`generarDatos()` del archivo original, que esta exportación reemplaza).

In [ ]:
def construir_payload_ticker(tk: str) -> dict:
    '''Construye el bloque JSON de un ticker con la forma exacta esperada por el frontend.'''
    seq = datos_secuencias[tk]

    # Fechas y precios de TODA la serie (incluye los WINDOW_SIZE días iniciales sin secuencia)
    fechas_iniciales = datos_features[tk]["Date"].iloc[:WINDOW_SIZE]
    precios_iniciales = datos_features[tk]["Close"].iloc[:WINDOW_SIZE]

    fechas_completas = pd.concat([fechas_iniciales, seq["fechas_full"]], ignore_index=True)
    precios_completos = pd.concat(
        [precios_iniciales.reset_index(drop=True), pd.Series(seq["precios_full"])],
        ignore_index=True,
    )

    modelos_payload = []
    for nombre_modelo, spec in MODEL_SPECS.items():
        m = resultados[tk][nombre_modelo]
        modelos_payload.append({
            "id": nombre_modelo,
            "arq": spec["arq"],
            "color": spec["color"],
            "acc": m["acc"],
            "prec": m["prec"],
            "rec": m["rec"],
            "f1": m["f1"],
            "accHist": m["accHist"],
            "senales": m["senales"],
            "senal": m["senal"],
            "conf": m["conf"],
            "confusion_matrix": m["confusion_matrix"],
        })

    return {
        "fechas": [f.strftime("%Y-%m-%d") for f in fechas_completas],
        "precios": [round(float(p), 4) for p in precios_completos],
        "modelos": modelos_payload,
    }


payload = {
    "metadata": {
        "sistema": "Ernesto Investing AI",
        "modulo": "Notebook 3 — Clasificadores RNN (LSTM, BiLSTM, GRU, SimpleRNN)",
        "generado_en": datetime.now().isoformat(timespec="seconds"),
        "fuente_datos": "Yahoo Finance (yfinance)",
        "periodo_descarga": PERIODO,
        "ventana_secuencia_dias": WINDOW_SIZE,
        "epocas": EPOCAS,
        "batch_size": BATCH_SIZE,
        "seed": SEED,
        "tickers": list(resultados.keys()),
    },
    "tickers": {tk: construir_payload_ticker(tk) for tk in resultados.keys()},
}

NOMBRE_ARCHIVO = "datos_rnns_clf.json"
with open(NOMBRE_ARCHIVO, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print(f"✅ Archivo '{NOMBRE_ARCHIVO}' exportado correctamente.")
print(f"   Tickers incluidos: {list(payload['tickers'].keys())}")
print(f"   Tamaño aproximado: {len(json.dumps(payload)) / 1024:.1f} KB")


✅ Archivo 'datos_rnns_clf.json' exportado correctamente.
   Tickers incluidos: ['FSM', 'VOLCABC1.LM', 'ABX.TO', 'BVN', 'BHP']
   Tamaño aproximado: 134.8 KB


## Módulo 11 · Validación del contrato de datos

Verificación rápida de que el JSON exportado cumple la estructura que el frontend espera leer
(claves y tipos correctos) antes de entregarlo junto con el resto de entregables.

In [ ]:
with open(NOMBRE_ARCHIVO, "r", encoding="utf-8") as f:
    payload_check = json.load(f)

assert "tickers" in payload_check, "Falta la clave 'tickers' en el JSON."
for tk, bloque in payload_check["tickers"].items():
    assert set(["fechas", "precios", "modelos"]).issubset(bloque.keys()), f"Faltan claves en {tk}"
    assert len(bloque["fechas"]) == len(bloque["precios"]), f"Longitud fechas/precios distinta en {tk}"
    assert len(bloque["modelos"]) == 4, f"{tk} no tiene las 4 arquitecturas"
    for modelo in bloque["modelos"]:
        for campo in ["id", "arq", "color", "acc", "prec", "rec", "f1", "accHist", "senales", "senal", "conf"]:
            assert campo in modelo, f"Falta el campo '{campo}' en el modelo {modelo.get('id')} de {tk}"
        assert len(modelo["accHist"]) == EPOCAS, f"accHist de {modelo['id']} en {tk} no tiene {EPOCAS} valores"
        assert len(modelo["senales"]) == len(bloque["fechas"]), f"senales de {modelo['id']} en {tk} mal alineadas"

print("✅ Validación superada: el JSON respeta el contrato de datos del frontend "
      "ernesto_investing_rnns_clasificador.html")

# Vista previa de un fragmento del JSON
print("\nEjemplo (primer ticker, primer modelo):")
primer_tk = list(payload_check["tickers"].keys())[0]
ejemplo = payload_check["tickers"][primer_tk]["modelos"][0].copy()
ejemplo["accHist"] = ejemplo["accHist"][:5] + ["..."]
ejemplo["senales"] = ejemplo["senales"][:5] + ["..."]
print(json.dumps(ejemplo, indent=2, ensure_ascii=False))


✅ Validación superada: el JSON respeta el contrato de datos del frontend ernesto_investing_rnns_clasificador.html

Ejemplo (primer ticker, primer modelo):
{
  "id": "LSTM",
  "arq": "LSTM(260)→Drop(0.2)→LSTM(130)→Dense(1)",
  "color": "#1565c0",
  "acc": 0.5169,
  "prec": 0.4808,
  "rec": 0.6098,
  "f1": 0.5376,
  "accHist": [
    0.4986,
    0.5432,
    0.5348,
    0.5348,
    0.5348,
    "..."
  ],
  "senales": [
    "BUY",
    "BUY",
    "BUY",
    "BUY",
    "BUY",
    "..."
  ],
  "senal": "SELL",
  "conf": 0.7084,
  "confusion_matrix": [
    [
      21,
      27
    ],
    [
      16,
      25
    ]
  ]
}


## Módulo 12 · Descarga del archivo JSON (opcional, en Google Colab)

Ejecutar la siguiente celda únicamente si se está trabajando en Google Colab, para descargar el
archivo `datos_rnns_clf.json` al equipo local y luego adjuntarlo entre los entregables.

In [ ]:
try:
    from google.colab import files
    files.download(NOMBRE_ARCHIVO)
except ImportError:
    print("Esta celda solo funciona dentro de Google Colab. "
          f"El archivo '{NOMBRE_ARCHIVO}' ya se encuentra guardado en el entorno de ejecución actual.")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
### 📌 Notas finales

- Este notebook **no** modifica el archivo `ernesto_investing_rnns_clasificador.html`. Ese paso
  corresponde a la **Fase 3** del trabajo (Capítulo 4.3), donde se reemplaza la función
  `generarDatos()` simulada por una llamada `fetch()` al endpoint `GET /api/rnns/{ticker}` de la
  API FastAPI (Fase 2), la cual a su vez puede leer este mismo archivo `datos_rnns_clf.json`.
- El archivo exportado debe entregarse junto con el notebook como parte del **Entregable 3**
  (Archivos JSON de Contrato de Datos).
- Si se desea reducir el tiempo de ejecución durante pruebas, se puede disminuir temporalmente
  `EPOCAS` en el Módulo 2 — recordando restaurarlo a 80 antes de la entrega final, conforme a las
  especificaciones del documento de trabajo.
